<a href="https://colab.research.google.com/github/Quang365/mri-recognition/blob/main/notebooks/VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [2]:
# Dataset paths
RANGPUR_DATA_ROOT = "/home/groups/comp3710/OASIS"

TRAIN_DIR = os.path.join(RANGPUR_DATA_ROOT, "keras_png_slices_train")
VAL_DIR = os.path.join(RANGPUR_DATA_ROOT, "keras_png_slices_validate")
TEST_DIR = os.path.join(RANGPUR_DATA_ROOT, "keras_png_slices_test")

print("Dataset root:", RANGPUR_DATA_ROOT)

if os.path.exists(RANGPUR_DATA_ROOT):
    print("OASIS dataset found.")
    print("Training directory:", TRAIN_DIR)
    print("Validation directory:", VAL_DIR)
    print("Test directory:", TEST_DIR)
else:
    print("OASIS dataset not available in this environment.")
    print("The dataset will be loaded when this notebook is run on Rangpur.")

Dataset root: /home/groups/comp3710/OASIS
OASIS dataset not available in this environment.
The dataset will be loaded when this notebook is run on Rangpur.


In [3]:
class MRIDataset(Dataset):
    """
    Dataset for loading preprocessed OASIS MRI slices.
    Each image is loaded as a single-channel grayscale tensor.
    """

    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform

        if os.path.exists(image_dir):
            self.image_files = sorted([
                f for f in os.listdir(image_dir)
                if f.lower().endswith(".png")
            ])
        else:
            self.image_files = []

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(
            self.image_dir,
            self.image_files[idx]
        )

        image = Image.open(image_path).convert("L")

        if self.transform:
            image = self.transform(image)

        return image

In [4]:
image_transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = MRIDataset(TRAIN_DIR, transform=image_transform)
val_dataset = MRIDataset(VAL_DIR, transform=image_transform)
test_dataset = MRIDataset(TEST_DIR, transform=image_transform)

print(f"Training images:   {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Testing images:    {len(test_dataset)}")

Training images:   0
Validation images: 0
Testing images:    0


In [5]:
LATENT_DIM = 16


class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()

        # Encoder: [1, 256, 256] -> [256, 8, 8]
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU()
        )

        self.flatten_dim = 256 * 8 * 8

        # Latent distribution parameters
        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)

        # Latent vector -> decoder feature map
        self.decoder_input = nn.Linear(
            latent_dim,
            self.flatten_dim
        )

        # Decoder: [256, 8, 8] -> [1, 256, 256]
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                256, 256, kernel_size=4, stride=2, padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                256, 128, kernel_size=4, stride=2, padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                128, 64, kernel_size=4, stride=2, padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                64, 32, kernel_size=4, stride=2, padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                32, 1, kernel_size=4, stride=2, padding=1
            ),
            nn.Sigmoid()
        )

    def encode(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, start_dim=1)

        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def decode(self, z):
        x = self.decoder_input(z)
        x = x.view(-1, 256, 8, 8)

        return self.decoder(x)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)

        return reconstruction, mu, logvar

In [7]:
def vae_loss(reconstruction, x, mu, logvar, beta=1.0):
    # Reconstruction loss
    recon_loss = F.mse_loss(
        reconstruction,
        x,
        reduction="sum"
    ) / x.size(0)

    # KL divergence
    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    ) / x.size(0)

    total_loss = recon_loss + beta * kl_loss

    return total_loss, recon_loss, kl_loss

In [10]:
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 30

# DataLoaders are created only when the OASIS dataset is available.
if len(train_dataset) > 0:
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Train batches:      {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")
    print(f"Test batches:       {len(test_loader)}")

else:
    train_loader = None
    val_loader = None
    test_loader = None

    print("OASIS dataset is not available in this runtime.")
    print("DataLoaders will be created when running on Rangpur.")


model = VAE(latent_dim=LATENT_DIM).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print()
print(f"Batch size:        {BATCH_SIZE}")
print(f"Learning rate:     {LEARNING_RATE}")
print(f"Epochs:            {NUM_EPOCHS}")
print(f"Latent dimensions: {LATENT_DIM}")

OASIS dataset is not available in this runtime.
DataLoaders will be created when running on Rangpur.

Batch size:        32
Learning rate:     0.001
Epochs:            30
Latent dimensions: 16


In [11]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    for images in loader:
        images = images.to(device, non_blocking=True)

        optimizer.zero_grad()

        reconstruction, mu, logvar = model(images)

        loss, recon_loss, kl_loss = vae_loss(
            reconstruction,
            images,
            mu,
            logvar
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    n_batches = len(loader)

    return (
        total_loss / n_batches,
        total_recon / n_batches,
        total_kl / n_batches
    )

In [12]:
@torch.no_grad()
def validate(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    for images in loader:
        images = images.to(device, non_blocking=True)

        reconstruction, mu, logvar = model(images)

        loss, recon_loss, kl_loss = vae_loss(
            reconstruction,
            images,
            mu,
            logvar
        )

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    n_batches = len(loader)

    return (
        total_loss / n_batches,
        total_recon / n_batches,
        total_kl / n_batches
    )

In [13]:
def train_vae(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=30,
    checkpoint_path="best_vae.pth"
):
    if train_loader is None or val_loader is None:
        print("Training skipped: OASIS dataset is not available.")
        return None

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_recon": [],
        "train_kl": [],
        "val_recon": [],
        "val_kl": []
    }

    best_val_loss = float("inf")

    for epoch in range(1, num_epochs + 1):

        train_loss, train_recon, train_kl = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device
        )

        val_loss, val_recon, val_kl = validate(
            model,
            val_loader,
            device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_recon"].append(train_recon)
        history["train_kl"].append(train_kl)
        history["val_recon"].append(val_recon)
        history["val_kl"].append(val_kl)

        print(
            f"Epoch [{epoch:02d}/{num_epochs}] | "
            f"Train: {train_loss:.2f} "
            f"(Recon: {train_recon:.2f}, KL: {train_kl:.2f}) | "
            f"Val: {val_loss:.2f} "
            f"(Recon: {val_recon:.2f}, KL: {val_kl:.2f})"
        )

        # Save the model with the best validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "latent_dim": LATENT_DIM
                },
                checkpoint_path
            )

            print(
                f"  -> Saved best model "
                f"(validation loss: {best_val_loss:.2f})"
            )

    return history

In [14]:
history = train_vae(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=NUM_EPOCHS,
    checkpoint_path="best_vae.pth"
)

Training skipped: OASIS dataset is not available.


In [15]:
@torch.no_grad()
def visualize_reconstructions(model, loader, device, num_images=6):
    if loader is None:
        print("Visualisation skipped: OASIS dataset is not available.")
        return

    model.eval()

    images = next(iter(loader))
    images = images[:num_images].to(device)

    reconstructions, _, _ = model(images)

    images = images.cpu()
    reconstructions = reconstructions.cpu()

    fig, axes = plt.subplots(
        2,
        num_images,
        figsize=(3 * num_images, 6)
    )

    for i in range(num_images):
        # Original MRI
        axes[0, i].imshow(
            images[i].squeeze(),
            cmap="gray"
        )
        axes[0, i].axis("off")

        # Reconstructed MRI
        axes[1, i].imshow(
            reconstructions[i].squeeze(),
            cmap="gray"
        )
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel(
        "Original",
        fontsize=12
    )

    axes[1, 0].set_ylabel(
        "Reconstruction",
        fontsize=12
    )

    plt.suptitle("Original vs VAE Reconstruction")
    plt.tight_layout()
    plt.show()

In [16]:
visualize_reconstructions(
    model,
    test_loader,
    device
)

Visualisation skipped: OASIS dataset is not available.


In [17]:
@torch.no_grad()
def extract_latent_features(model, loader, device):
    if loader is None:
        print("Latent extraction skipped: OASIS dataset is not available.")
        return None

    model.eval()

    latent_vectors = []

    for images in loader:
        images = images.to(device, non_blocking=True)

        mu, _ = model.encode(images)

        latent_vectors.append(mu.cpu())

    return torch.cat(latent_vectors, dim=0).numpy()

In [18]:
from sklearn.decomposition import PCA


def visualize_latent_manifold(model, loader, device):
    latent_vectors = extract_latent_features(
        model,
        loader,
        device
    )

    if latent_vectors is None:
        return

    print("Latent representation shape:", latent_vectors.shape)

    pca = PCA(n_components=2)
    latent_2d = pca.fit_transform(latent_vectors)

    plt.figure(figsize=(8, 6))

    plt.scatter(
        latent_2d[:, 0],
        latent_2d[:, 1],
        s=10,
        alpha=0.6
    )

    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title("2D Manifold of VAE Latent Representations")
    plt.grid(alpha=0.2)

    plt.show()

    print(
        "Explained variance ratio:",
        pca.explained_variance_ratio_
    )

In [19]:
visualize_latent_manifold(
    model,
    test_loader,
    device
)

Latent extraction skipped: OASIS dataset is not available.
